In [1]:
import pandas as pd
import math
import os
import hashlib

In [2]:
def hash_dataframe(df):
    """Devuelve un hash único para un dataframe dado (ignorando el orden de filas)."""
    df_sorted = df.sort_values(by=df.columns.tolist()).reset_index(drop=True)
    return hashlib.md5(pd.util.hash_pandas_object(df_sorted, index=True).values).hexdigest()

In [3]:
os.makedirs('training', exist_ok=True)

In [4]:
global_df = pd.read_excel('global_table_con_UID.xlsx', sheet_name='Table')
test_groups = [pd.read_excel('global_table_con_UID.xlsx', sheet_name=f'Test{i+1:02d}') for i in range(10)]

In [5]:
for i in range(1, 11):
    print(f"Generando entrenamiento {i:02d}...")

    # Excluir datos de test y formar conjunto de entrenamiento
    test_uids = test_groups[i - 1]['UID'].unique()
    train_df = global_df[~global_df['UID'].isin(test_uids)].copy()

    # Obtener UIDs únicos
    uids = sorted(train_df['UID'].unique())
    total_uids = len(uids)
    tamano_subgrupo = math.ceil(0.9 * total_uids)

    # Crear archivo Excel por entrenamiento
    excel_path = f"training/Train{i:02d}.xlsx"
    hashes = set()

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        # Guardar todos los datos de entrenamiento
        train_df.to_excel(writer, sheet_name=f"Train{i:02d}_All", index=False)

        # Crear 10 subconjuntos con verificación
        for j in range(10):
            start = j % total_uids
            uids_reordenados = uids[start:] + uids[:start]
            uids_sub = uids_reordenados[:tamano_subgrupo]
            df_sub = train_df[train_df['UID'].isin(uids_sub)].copy()

            # Verificar que el subconjunto no sea duplicado
            hash_val = hash_dataframe(df_sub)
            if hash_val in hashes:
                raise ValueError(f"Subgrupo duplicado detectado en entrenamiento {i:02d}, subgrupo {j+1:02d}")
            hashes.add(hash_val)

            # Guardar el subconjunto
            nombre_hoja = f"Train{i:02d}_Sub{j+1:02d}"
            df_sub.to_excel(writer, sheet_name=nombre_hoja[:31], index=False)

    print(f"Entrenamiento {i:02d} guardado en '{excel_path}' con subconjuntos únicos.")

Generando entrenamiento 01...
Entrenamiento 01 guardado en 'training/Train01.xlsx' con subconjuntos únicos.
Generando entrenamiento 02...
Entrenamiento 02 guardado en 'training/Train02.xlsx' con subconjuntos únicos.
Generando entrenamiento 03...
Entrenamiento 03 guardado en 'training/Train03.xlsx' con subconjuntos únicos.
Generando entrenamiento 04...
Entrenamiento 04 guardado en 'training/Train04.xlsx' con subconjuntos únicos.
Generando entrenamiento 05...
Entrenamiento 05 guardado en 'training/Train05.xlsx' con subconjuntos únicos.
Generando entrenamiento 06...
Entrenamiento 06 guardado en 'training/Train06.xlsx' con subconjuntos únicos.
Generando entrenamiento 07...
Entrenamiento 07 guardado en 'training/Train07.xlsx' con subconjuntos únicos.
Generando entrenamiento 08...
Entrenamiento 08 guardado en 'training/Train08.xlsx' con subconjuntos únicos.
Generando entrenamiento 09...
Entrenamiento 09 guardado en 'training/Train09.xlsx' con subconjuntos únicos.
Generando entrenamiento 10..